In [1]:
import json
import os
import time
from pathlib import Path
from typing import Any

import pandas as pd
from dotenv import load_dotenv
from minsearch import Index
from openai import OpenAI
from tqdm.auto import tqdm

In [2]:
load_dotenv("../.env")

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
GENERATION_MODEL = os.getenv(
    "OPENAI_MODEL",
    "gpt-4o-mini",
)
JUDGE_MODEL = os.getenv(
    "OPENAI_JUDGE_MODEL",
    GENERATION_MODEL,
)

if not OPENAI_API_KEY:
    raise RuntimeError("OPENAI_API_KEY is missing.")

client = OpenAI(api_key=OPENAI_API_KEY)

print("Generation model:", GENERATION_MODEL)
print("Judge model:", JUDGE_MODEL)

Generation model: gpt-4o-mini
Judge model: gpt-4o-mini


In [3]:
DATA_DIR = Path("../data")

df_documents = pd.read_csv(
    DATA_DIR / "knowledge_base.csv"
).fillna("")

df_questions = pd.read_csv(
    DATA_DIR / "ground-truth-retrieval.csv"
).fillna("")

print("Documents:", df_documents.shape)
print("Questions:", df_questions.shape)

df_questions.head()

Documents: (10, 7)
Questions: (30, 2)


,id,question
0,policy-001,What is the cancellation policy for private le...
1,policy-001,How much notice do I need to give to cancel a ...
2,policy-001,Will I get a refund if I cancel my lesson late?
3,policy-002,How long can I book a court for my practice se...
4,policy-002,What time should I arrive before my court book...


In [4]:
documents = df_documents.to_dict(orient="records")

text_fields = [
    "category",
    "title",
    "content",
    "coach_name",
    "skill_level",
    "location",
]

index = Index(
    text_fields=text_fields,
    keyword_fields=["id"],
)

index.fit(documents)

In [5]:
BOOST_PATH = DATA_DIR / "best-minsearch-boost.json"

if BOOST_PATH.exists():
    with open(
        BOOST_PATH,
        "r",
        encoding="utf-8",
    ) as file:
        BEST_BOOST = json.load(file)
else:
    BEST_BOOST = {
        "title": 2.0,
        "content": 1.0,
        "coach_name": 1.5,
        "category": 1.2,
        "skill_level": 1.0,
        "location": 0.5,
    }

BEST_BOOST

{'category': 2.6765387031145362,
 'title': 0.3477553305176646,
 'content': 1.2657654590558112,
 'coach_name': 0.11918887775228137,
 'skill_level': 0.6559139244108101,
 'location': 1.0107105762067248}

In [6]:
def search(
    query: str,
    num_results: int = 5,
) -> list[dict[str, Any]]:
    return index.search(
        query=query,
        filter_dict={},
        boost_dict=BEST_BOOST,
        num_results=num_results,
    )

In [7]:
baseline_prompt_template = """
You are CourtMate, an assistant for a badminton club.

Answer the QUESTION using only the information in the CONTEXT.

If the answer is not available in the context, say that the
information is not available.

QUESTION:
{question}

CONTEXT:
{context}
""".strip()

In [8]:
document_template = """
Document ID: {id}
Category: {category}
Title: {title}
Content: {content}
Coach: {coach_name}
Skill level: {skill_level}
Location: {location}
""".strip()

In [9]:
def build_prompt(
    question: str,
    search_results: list[dict],
    prompt_template: str,
) -> str:
    context = "\n\n".join(
        document_template.format(**document)
        for document in search_results
    )

    return prompt_template.format(
        question=question,
        context=context,
    )

In [10]:
def llm(
    prompt: str,
    model: str,
) -> str:
    response = client.responses.create(
        model=model,
        input=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
    )

    return response.output_text.strip()

In [11]:
def rag(
    question: str,
    prompt_template: str,
    model: str = GENERATION_MODEL,
    num_results: int = 5,
) -> dict[str, Any]:
    search_results = search(
        query=question,
        num_results=num_results,
    )

    prompt = build_prompt(
        question=question,
        search_results=search_results,
        prompt_template=prompt_template,
    )

    answer = llm(
        prompt=prompt,
        model=model,
    )

    return {
        "answer": answer,
        "retrieved_ids": [
            document["id"]
            for document in search_results
        ],
        "model": model,
    }

In [12]:
result = rag(
    question="How early should I cancel a private lesson?",
    prompt_template=baseline_prompt_template,
)

result

{'answer': 'You should cancel a private lesson at least 24 hours before the scheduled start time.',
 'retrieved_ids': ['course-001',
  'course-002',
  'policy-001',
  'coach-001',
  'faq-002'],
 'model': 'gpt-4o-mini'}

In [13]:
SAMPLE_SIZE = min(
    50,
    len(df_questions),
)

In [14]:
df_sample = df_questions.sample(
    n=SAMPLE_SIZE,
    random_state=42,
).reset_index(drop=True)

df_sample.head()

,id,question
0,faq-002,What should I bring with me for my badminton s...
1,course-001,What is a private lesson in badminton?
2,facility-001,Can I rent equipment for badminton at the faci...
3,course-001,Where are the private lessons held?
4,dropin-001,Where is the beginner drop-in held?


In [15]:
judge_prompt_template = """
You are evaluating an answer produced by a badminton club
retrieval-augmented generation system.

Evaluate how well the GENERATED ANSWER responds to the USER QUESTION.

Use one of these labels:

RELEVANT:
- The answer directly addresses the question.
- The answer is clear and sufficiently complete.
- It does not contain unsupported or contradictory claims.

PARTLY_RELEVANT:
- The answer contains useful and mostly correct information.
- However, it is incomplete, vague, indirect, or contains a minor
  unsupported detail.

NON_RELEVANT:
- The answer does not answer the question.
- The answer is substantially incorrect.
- The answer invents important facts.
- The answer contradicts the available information.

Return valid JSON only, without Markdown code fences.

Required JSON format:

{{
  "relevance": "RELEVANT | PARTLY_RELEVANT | NON_RELEVANT",
  "explanation": "Brief explanation of the rating"
}}

USER QUESTION:
{question}

GENERATED ANSWER:
{answer}
""".strip()

In [16]:
VALID_RELEVANCE_LABELS = {
    "RELEVANT",
    "PARTLY_RELEVANT",
    "NON_RELEVANT",
}


def clean_json_text(text: str) -> str:
    text = text.strip()

    if text.startswith("```json"):
        text = text[len("```json"):]

    if text.startswith("```"):
        text = text[len("```"):]

    if text.endswith("```"):
        text = text[:-len("```")]

    return text.strip()


def parse_judge_response(text: str) -> dict[str, str]:
    cleaned = clean_json_text(text)
    parsed = json.loads(cleaned)

    relevance = str(
        parsed.get("relevance", "")
    ).strip().upper()

    explanation = str(
        parsed.get("explanation", "")
    ).strip()

    if relevance not in VALID_RELEVANCE_LABELS:
        raise ValueError(
            f"Invalid relevance label: {relevance}"
        )

    return {
        "relevance": relevance,
        "explanation": explanation,
    }

In [17]:
def evaluate_answer_with_judge(
    question: str,
    answer: str,
    judge_model: str = JUDGE_MODEL,
) -> dict[str, str]:
    prompt = judge_prompt_template.format(
        question=question,
        answer=answer,
    )

    raw_response = llm(
        prompt=prompt,
        model=judge_model,
    )

    return parse_judge_response(raw_response)

In [18]:
judge_result = evaluate_answer_with_judge(
    question="How early should I cancel a private lesson?",
    answer=(
        "Private lessons should be cancelled at least "
        "24 hours before the scheduled start time."
    ),
)

judge_result

{'relevance': 'RELEVANT',
 'explanation': 'The answer directly addresses the question by providing a specific time frame for cancellation, which is clear and complete without any unsupported claims.'}

In [19]:
def evaluate_rag_configuration(
    df_eval_questions: pd.DataFrame,
    configuration_name: str,
    prompt_template: str,
    generation_model: str,
    judge_model: str,
    num_results: int = 5,
) -> pd.DataFrame:
    rows = []

    for record in tqdm(
        df_eval_questions.to_dict(orient="records")
    ):
        question = record["question"]
        expected_document_id = record["id"]

        try:
            rag_result = rag(
                question=question,
                prompt_template=prompt_template,
                model=generation_model,
                num_results=num_results,
            )

            judge_result = evaluate_answer_with_judge(
                question=question,
                answer=rag_result["answer"],
                judge_model=judge_model,
            )

            rows.append(
                {
                    "configuration": configuration_name,
                    "question": question,
                    "expected_document_id": (
                        expected_document_id
                    ),
                    "answer": rag_result["answer"],
                    "retrieved_ids": json.dumps(
                        rag_result["retrieved_ids"]
                    ),
                    "relevance": (
                        judge_result["relevance"]
                    ),
                    "explanation": (
                        judge_result["explanation"]
                    ),
                    "generation_model": generation_model,
                    "judge_model": judge_model,
                    "error": "",
                }
            )

        except Exception as exc:
            rows.append(
                {
                    "configuration": configuration_name,
                    "question": question,
                    "expected_document_id": (
                        expected_document_id
                    ),
                    "answer": "",
                    "retrieved_ids": "[]",
                    "relevance": "ERROR",
                    "explanation": "",
                    "generation_model": generation_model,
                    "judge_model": judge_model,
                    "error": str(exc),
                }
            )

        time.sleep(0.2)

    return pd.DataFrame(rows)

In [20]:
df_baseline_eval = evaluate_rag_configuration(
    df_eval_questions=df_sample,
    configuration_name="baseline_prompt",
    prompt_template=baseline_prompt_template,
    generation_model=GENERATION_MODEL,
    judge_model=JUDGE_MODEL,
    num_results=5,
)

  0%|          | 0/30 [00:00<?, ?it/s]

In [21]:
df_baseline_eval.to_csv(
    DATA_DIR / "rag-evaluation-baseline.csv",
    index=False,
)

In [22]:
df_baseline_eval["relevance"].value_counts()

relevance
RELEVANT        29
NON_RELEVANT     1
Name: count, dtype: int64

In [23]:
baseline_distribution = (
    df_baseline_eval["relevance"]
    .value_counts(normalize=True)
    .rename("proportion")
    .reset_index()
)

baseline_distribution

,relevance,proportion
0,RELEVANT,0.966667
1,NON_RELEVANT,0.033333


In [24]:
df_baseline_eval[
    df_baseline_eval["relevance"] == "NON_RELEVANT"
][
    [
        "question",
        "answer",
        "retrieved_ids",
        "explanation",
    ]
]

,question,answer,retrieved_ids,explanation
12,Are there any specific time slots available fo...,The information is not available.,"[""policy-002"", ""faq-001"", ""facility-001"", ""dro...",The answer does not address the user's questio...
